# Genetic Algorithm (From Scratch) — Toy Example

_Generated: 2025-10-22T19:51:49.943079Z_

This notebook implements a **Genetic Algorithm (GA) from first principles** using only NumPy. We evolve a population of real‑valued vectors to optimize a **non‑convex 2D function (Rastrigin)**. The GA uses **tournament selection**, **BLX‑α crossover**, **Gaussian mutation**, and **elitism**. We visualize convergence and the search on the function landscape, and we save the best solution and logs.

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess, math
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports & Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)
SEED = 42
rng = np.random.default_rng(SEED)

## 2) Toy Objective: 2D Rastrigin Function (maximize -Rastrigin)

Rastrigin (for dimension 2) is a standard **multimodal** benchmark. We will **maximize** `-Rastrigin(x)`.

$$\text{Rastrigin}(\mathbf{x}) = A d + \sum_{i=1}^d[x_i^2 - A\cos(2\pi x_i)],\quad A=10$$
Global minimum is at `x=0`, value `0`. Thus our **maximum** of `-Rastrigin` is `0` at `x=(0,0)`. We search in the box **[-5.12, 5.12]^2**.

In [ ]:
A = 10.0
dim = 2
bounds = np.array([[-5.12, 5.12], [-5.12, 5.12]], dtype=float)

def rastrigin(x):
    x = np.asarray(x, dtype=float)
    return A*len(x) + np.sum(x*x - A*np.cos(2*np.pi*x))

def fitness(x):
    # maximize -rastrigin
    return -rastrigin(x)

def sample_uniform(n):
    lo = bounds[:,0]; hi = bounds[:,1]
    return rng.uniform(lo, hi, size=(n, dim))

## 3) GA Building Blocks — Selection, Crossover, Mutation, Elitism

In [ ]:
def tournament_select(pop, fit, k=3):
    # Return index of tournament winner among k random individuals (higher fitness wins).
    idx = rng.choice(len(pop), size=k, replace=False)
    best = idx[0]
    for i in idx[1:]:
        if fit[i] > fit[best]:
            best = i
    return best

def blx_alpha_crossover(p1, p2, alpha=0.5):
    # BLX-α: sample each gene from [min-α*I, max+α*I], I = interval between parents.
    lo = np.minimum(p1, p2)
    hi = np.maximum(p1, p2)
    I = hi - lo
    lo_ext = lo - alpha * I
    hi_ext = hi + alpha * I
    child = rng.uniform(lo_ext, hi_ext)
    return np.clip(child, bounds[:,0], bounds[:,1])

def gaussian_mutation(x, sigma=0.1, p=0.2):
    # Per-gene Gaussian mutation with probability p; clip to bounds.
    mask = rng.random(dim) < p
    noise = rng.normal(0, sigma, size=dim)
    x_new = x.copy()
    x_new[mask] += noise[mask]
    return np.clip(x_new, bounds[:,0], bounds[:,1])

## 4) Putting It Together — The GA Loop

In [ ]:
from dataclasses import dataclass

@dataclass
class GAConfig:
    pop_size:int = 60
    gens:int = 120
    tournament:int = 3
    alpha:float = 0.5          # BLX-α
    mut_sigma:float = 0.15
    mut_prob:float = 0.2
    elite:int = 4
    seed:int = SEED

CFG = GAConfig()

def evaluate(pop):
    return np.array([fitness(ind) for ind in pop], dtype=float)

def evolve(cfg=CFG, verbose=True):
    rng_local = np.random.default_rng(cfg.seed)
    global rng
    rng = rng_local  # use local seed

    pop = sample_uniform(cfg.pop_size)
    fit = evaluate(pop)

    best_hist = []
    mean_hist = []
    snap_pop = {}

    for g in range(cfg.gens):
        order = np.argsort(-fit)
        pop = pop[order]
        fit = fit[order]

        best_hist.append(float(fit[0]))
        mean_hist.append(float(np.mean(fit)))

        if verbose and (g % 10 == 0 or g == cfg.gens-1):
            print(f"Gen {g:03d} | best={fit[0]:.4f} | mean={np.mean(fit):.4f} | best_x={pop[0]}")

        # Keep snapshots (start, mid, end)
        if g in [0, cfg.gens//2, cfg.gens-1]:
            snap_pop[g] = pop.copy()

        # Elitism
        new_pop = [pop[i].copy() for i in range(cfg.elite)]

        # Create offspring
        while len(new_pop) < cfg.pop_size:
            i = tournament_select(pop, fit, cfg.tournament)
            j = tournament_select(pop, fit, cfg.tournament)
            c = blx_alpha_crossover(pop[i], pop[j], cfg.alpha)
            c = gaussian_mutation(c, sigma=cfg.mut_sigma, p=cfg.mut_prob)
            new_pop.append(c)

        pop = np.array(new_pop)
        fit = evaluate(pop)

    # final sort
    order = np.argsort(-fit)
    pop = pop[order]; fit = fit[order]

    return pop, fit, np.array(best_hist), np.array(mean_hist), snap_pop

pop, fit, best_hist, mean_hist, snap_pop = evolve(CFG, verbose=True)
best_x, best_f = pop[0], fit[0]
print("Best found:", best_x, "fitness:", best_f)

## 5) Convergence Diagnostics

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.plot(best_hist, label="best")
plt.plot(mean_hist, label="mean")
plt.xlabel("Generation"); plt.ylabel("Fitness (maximize)")
plt.title("GA Convergence on -Rastrigin (2D)")
plt.legend(); plt.tight_layout(); plt.show()

## 6) Landscape & Population Snapshots

In [ ]:
# Draw the Rastrigin landscape and overlay population at start/mid/end
res = 200
xs = np.linspace(bounds[0,0], bounds[0,1], res)
ys = np.linspace(bounds[1,0], bounds[1,1], res)
XX, YY = np.meshgrid(xs, ys)
ZZ = np.zeros_like(XX)
for i in range(res):
    for j in range(res):
        ZZ[i,j] = fitness(np.array([XX[i,j], YY[i,j]]))

def plot_pop(ax, P, title):
    cs = ax.contourf(XX, YY, ZZ, levels=30)
    ax.scatter(P[:,0], P[:,1], s=12, c='k')
    ax.set_title(title)
    ax.set_xlim(bounds[0,0], bounds[0,1])
    ax.set_ylim(bounds[1,0], bounds[1,1])

fig, axes = plt.subplots(1,3, figsize=(12,4))
gens = sorted(snap_pop.keys())
titles = [f"Generation {g}" for g in gens]
for ax, g, t in zip(axes, gens, titles):
    plot_pop(ax, snap_pop[g], t)
plt.tight_layout(); plt.show()

print("Best solution:", best_x, " -> f =", best_f, " (global optimum is at (0,0) with f=0).")

## 7) Optional: Multiple Seeds (robustness)

In [ ]:
def run_seeds(n=5):
    bests = []
    for s in range(n):
        cfg = GAConfig(seed=SEED + s)
        pop, fit, bh, mh, _ = evolve(cfg, verbose=False)
        bests.append(fit[0])
    return np.array(bests)

bests = run_seeds(5)
print("Best fitness across 5 seeds:", bests, "mean=", bests.mean(), "std=", bests.std())

## 8) Save Artifacts & Download

In [ ]:
import os, json as _json
os.makedirs("artifacts", exist_ok=True)

summary = {
    "best_x": best_x.tolist(),
    "best_fitness": float(best_f),
    "config": {k: (int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, (np.floating,)) else v)
               for k,v in CFG.__dict__.items()}
}
with open("artifacts/ga_summary.json","w") as f:
    _json.dump(summary, f, indent=2)

np.savez("artifacts/ga_logs_and_population.npz",
         best_hist=best_hist, mean_hist=mean_hist,
         final_pop=pop, final_fit=fit)

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 9) Exercises & Extensions

- Replace BLX‑α with **simulated binary crossover (SBX)** and compare.
- Make mutation **self‑adaptive** (e.g., evolve `σ` per gene).
- Add **constraint handling** (repair, penalty) and try other problems (Ackley, Rosenbrock).
- Try a **binary GA** on 0/1 knapsack or 8‑Queens to contrast representations.
- Implement a **steady‑state GA** and compare convergence profiles.
